# Cellpose-SAM nuclear segmentation — Stroke MERSCOPE DAPI

Segments nuclei from the full-resolution MERSCOPE **DAPI mosaics** in `/Users/christoffer/Downloads/dapi`
(one `.tif` per region, organised by slide) using **Cellpose-SAM** (cellpose v4, `cpsam` model) on the M4 GPU
(MPS).

The nuclei produced here are meant to become a **prior for Baysor** (`prior_segmentation`): real nuclear
seeds anchor the transcript-based segmentation, which matters a lot in the infarct where transcript density
alone is misleading.

### Why this notebook is built the way it is
These mosaics are **gigantic** (e.g. 54121 × 57752 ≈ 3.1 Gpx, ~6 GB each, ~89 GB total) and strip-compressed
(not internally tiled). So we:
1. **Stream + downsample** each mosaic in row-bands (bounded memory) to a working resolution where nuclei are
   ~30 px across — Cellpose-SAM's sweet spot.
2. Run Cellpose-SAM (which tiles internally) on the downsampled image.
3. Save the label mask (working res), per-nucleus centroids/areas **scaled back to full-res pixels**, and a QC
   overlay — per region, skip-if-done.

> **Coordinate note (read before Stage 2):** masks are in *mosaic pixel* space. To use them as a Baysor prior
> they must be mapped to your transcripts' *micron* space via the MERSCOPE
> `micron_to_mosaic_pixel_transform.csv`. That file is **not** in the DAPI folder — see the last section.

**Kernel:** `Python (sopa)` (cellpose 4.0.8, torch 2.9 + MPS).

In [ ]:
import os, json, time, glob
from pathlib import Path
import numpy as np
import tifffile, zarr
import torch
from skimage.measure import block_reduce, regionprops_table
from skimage.segmentation import find_boundaries
import matplotlib.pyplot as plt
from cellpose import models

print("torch", torch.__version__, "| mps", torch.backends.mps.is_available())
print("devices: mps available ->", torch.backends.mps.is_available())

## 1. Configuration

In [ ]:
DAPI_ROOT = Path("/Users/christoffer/Downloads/dapi")
OUT_ROOT  = Path("/Users/christoffer/Downloads/dapi_nuclei")   # masks + centroids land here
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# --- imaging geometry -------------------------------------------------------
PIXEL_SIZE_UM      = 0.108   # MERSCOPE mosaic pixel size (CONFIRM against your instrument!)
EXPECTED_NUCLEUS_UM = 10.0   # rough nuclear diameter in microns
TARGET_DIAM_PX     = 30      # Cellpose-SAM works best with nuclei ~30 px across

# Downsample factor so a nucleus lands near TARGET_DIAM_PX at working resolution.
DOWNSAMPLE = max(1, round((EXPECTED_NUCLEUS_UM / TARGET_DIAM_PX) / PIXEL_SIZE_UM))
WORKING_UM_PER_PX = PIXEL_SIZE_UM * DOWNSAMPLE
DIAM_PX = EXPECTED_NUCLEUS_UM / WORKING_UM_PER_PX

# --- Cellpose-SAM parameters (tune in section 3) ----------------------------
FLOW_THRESHOLD     = 0.4     # higher -> more (incl. lower-quality) masks
CELLPROB_THRESHOLD = 0.0     # lower  -> more / larger masks; raise to be stricter
MIN_SIZE_PX        = 50      # drop objects smaller than this (working-res pixels)
BATCH_SIZE         = 8

# --- device -----------------------------------------------------------------
# MPS is fastest on this Mac; flip to 'cpu' if you hit MPS instability.
DEVICE = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

tifs = sorted(glob.glob(str(DAPI_ROOT / "*" / "*.tif")))
print(f"DOWNSAMPLE={DOWNSAMPLE}  ->  working {WORKING_UM_PER_PX:.3f} um/px, nucleus ~{DIAM_PX:.0f} px")
print(f"device = {DEVICE}")
print(f"found {len(tifs)} DAPI mosaics across {len(set(Path(f).parent.name for f in tifs))} slides")

## 2. Helpers — streaming downsample, model, segmentation

`read_downsampled` streams the strip-compressed mosaic in row-bands and mean-pools each band by
`DOWNSAMPLE`, so we never hold the full 6 GB image in memory.

In [ ]:
def read_downsampled(path, f, band_rows=4096):
    """Mean-pool a huge strip TIFF by integer factor f, streaming in row-bands."""
    store = tifffile.imread(path, aszarr=True)
    try:
        z = zarr.open(store, mode="r")
        H, W = z.shape
        Wc = (W // f) * f                      # crop to a multiple of f
        out_rows = []
        step = (band_rows // f) * f
        for y0 in range(0, (H // f) * f, step):
            y1 = min(y0 + step, (H // f) * f)
            band = np.asarray(z[y0:y1, :Wc])
            # reshape-pool (rows//f, f, cols//f, f) -> mean over the two f axes
            r = (y1 - y0) // f
            band = band[: r * f].reshape(r, f, Wc // f, f).mean(axis=(1, 3))
            out_rows.append(band.astype(np.float32))
        img = np.concatenate(out_rows, axis=0)
        return img
    finally:
        store.close()


model = models.CellposeModel(gpu=True, device=DEVICE)
print("loaded Cellpose-SAM:", model.pretrained_model)

In [ ]:
def segment_dapi(img):
    """Run Cellpose-SAM on a 2D working-resolution DAPI image -> int32 label mask."""
    masks, flows, styles = model.eval(
        img,
        diameter=DIAM_PX,
        flow_threshold=FLOW_THRESHOLD,
        cellprob_threshold=CELLPROB_THRESHOLD,
        min_size=MIN_SIZE_PX,
        batch_size=BATCH_SIZE,
        normalize=True,             # internal 1-99 percentile normalisation
    )
    return masks.astype(np.int32)


def nuclei_table(masks, downsample):
    """Per-nucleus centroid (scaled to FULL-RES pixels) + area."""
    props = regionprops_table(masks, properties=("label", "centroid", "area"))
    import pandas as pd
    df = pd.DataFrame(props).rename(columns={
        "centroid-0": "y_work", "centroid-1": "x_work", "area": "area_work_px"})
    df["x_fullres_px"] = df["x_work"] * downsample
    df["y_fullres_px"] = df["y_work"] * downsample
    df["area_fullres_px"] = df["area_work_px"] * downsample**2
    return df

## 3. Tune on one region (do this first!)

Pick a representative region, segment a dense crop, and eyeball the overlay. Adjust `CELLPROB_THRESHOLD`
(↑ = fewer/cleaner), `FLOW_THRESHOLD`, and `DOWNSAMPLE` until nuclei look right **before** launching the batch.
Stroke infarct cores are densely packed — that's the hardest place, so crop there if you can.

In [ ]:
TUNE_TIF = tifs[0]
print("tuning on:", Path(TUNE_TIF).parent.name, "/", Path(TUNE_TIF).name)

work = read_downsampled(TUNE_TIF, DOWNSAMPLE)
print("working image:", work.shape, "(", round(work.nbytes/1e9, 2), "GB )")

# take a central 1024x1024 working-res crop
H, W = work.shape
cs = 1024
y0, x0 = max(0, H//2 - cs//2), max(0, W//2 - cs//2)
crop = work[y0:y0+cs, x0:x0+cs]

t0 = time.time()
m = segment_dapi(crop)
print(f"crop segmented: {m.max()} nuclei in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
vmax = np.percentile(crop, 99)
ax[0].imshow(crop, cmap="gray", vmax=vmax); ax[0].set_title("DAPI (crop)")
ax[1].imshow(crop, cmap="gray", vmax=vmax)
ax[1].imshow(np.ma.masked_where(~find_boundaries(m), m > 0), cmap="autumn", alpha=0.9)
ax[1].set_title(f"Cellpose-SAM: {m.max()} nuclei  (cellprob={CELLPROB_THRESHOLD}, flow={FLOW_THRESHOLD})")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 4. Full-image runner

Segments a whole mosaic at working resolution and writes, per region, into `OUT_ROOT/<slide>/<sample>/`:
`nuclei_labels.tif` (working-res mask), `nuclei.parquet` (centroids in full-res px), `overlay.png` (QC),
`meta.json` (shapes / downsample / params). Already-finished regions are skipped.

In [ ]:
def out_dir_for(path):
    p = Path(path)
    d = OUT_ROOT / p.parent.name / p.stem
    d.mkdir(parents=True, exist_ok=True)
    return d

def is_done(path):
    d = OUT_ROOT / Path(path).parent.name / Path(path).stem
    return (d / "nuclei.parquet").exists() and (d / "meta.json").exists()

def process_region(path):
    d = out_dir_for(path)
    work = read_downsampled(path, DOWNSAMPLE)
    masks = segment_dapi(work)
    df = nuclei_table(masks, DOWNSAMPLE)

    tifffile.imwrite(d / "nuclei_labels.tif", masks, compression="zlib")
    df.to_parquet(d / "nuclei.parquet")

    # QC overlay (downsample again for a light PNG)
    q = 4
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(work[::q, ::q], cmap="gray", vmax=np.percentile(work, 99))
    ax.imshow(np.ma.masked_where(~find_boundaries(masks[::q, ::q]), masks[::q, ::q] > 0),
              cmap="autumn", alpha=0.8)
    ax.set_title(f"{Path(path).stem}: {len(df)} nuclei"); ax.axis("off")
    fig.savefig(d / "overlay.png", dpi=120, bbox_inches="tight"); plt.close(fig)

    meta = dict(sample=Path(path).stem, slide=Path(path).parent.name,
                source_tif=str(path), work_shape=list(map(int, work.shape)),
                downsample=DOWNSAMPLE, pixel_size_um=PIXEL_SIZE_UM,
                working_um_per_px=WORKING_UM_PER_PX, n_nuclei=int(len(df)),
                cellprob_threshold=CELLPROB_THRESHOLD, flow_threshold=FLOW_THRESHOLD,
                diam_px=float(DIAM_PX), min_size_px=MIN_SIZE_PX)
    (d / "meta.json").write_text(json.dumps(meta, indent=2))
    return len(df)

## 5. Batch all regions

> ⚠️ Heavy & long — 50 mosaics on MPS. Resumable (finished regions skipped). Run one region first to time it,
> then let the rest go.

In [ ]:
results = []
for i, path in enumerate(tifs, 1):
    name = f"{Path(path).parent.name}/{Path(path).stem}"
    if is_done(path):
        print(f"[{i}/{len(tifs)}] skip (done): {name}")
        continue
    print(f"[{i}/{len(tifs)}] {name} ...", end=" ", flush=True)
    t0 = time.time()
    try:
        n = process_region(path)
        print(f"{n} nuclei  ({time.time()-t0:.0f}s)")
        results.append((name, "ok", n))
    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")
        results.append((name, "error", str(e)))

ok = [r for r in results if r[1] == "ok"]
print(f"\nDone this run: {len(ok)} ok, {len(results)-len(ok)} errors")

## 6. Next step — mapping nuclei to microns for the Baysor prior

The masks/centroids above are in **mosaic pixel** space. Your Baysor transcripts live in **microns**
(`global_x`, `global_y`). To connect them you need the MERSCOPE affine
`micron_to_mosaic_pixel_transform.csv` (a 3×3 matrix in the MERSCOPE `images/` output folder), which converts
micron coordinates to mosaic pixels:

```
[px_x, px_y, 1]^T  =  M @ [um_x, um_y, 1]^T
```

**That file is not in `/Users/christoffer/Downloads/dapi`.** Once we locate it (per region), the next notebook
will, for each region:
1. load the transcripts CSV and `M`,
2. map each transcript `global_x/global_y` → mosaic px → working px (`÷ DOWNSAMPLE`),
3. sample `nuclei_labels.tif` at that location → nucleus id per transcript (a `nucleus` prior column),
4. run Baysor with `prior_segmentation=:nucleus`, `prior_segmentation_confidence≈0.5`.

👉 **Where is the full MERSCOPE output** (the `images/` folder with `micron_to_mosaic_pixel_transform.csv`)
for these regions? With that I'll wire up the prior; without it we'd have to approximate the transform from
transcript/image bounding boxes, which is risky for a prior.